[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Egocentric Bottle Action Detection AI**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
An edge-ready computer vision system built on **YOLO11** to detect bottle opening and closing actions from a first-person perspective.  
The model combines:
- Instance segmentation for bottles and caps  
- 21-point hand pose tracking  

in a single inference pipeline.

---

## Core Workflow

### Tracking  
Uses a unified **YOLO11-seg-pose** model to detect objects and hand keypoints simultaneously.

### Intent Logic  
Tracks fingertip rotation between the thumb and index finger to detect twisting direction.

### Trigger Mechanism  
- **Opening** → Anticlockwise rotation  
- **Closing** → Clockwise rotation  
- **Idle** → No twisting activity for 0.6 seconds  

The system ignores grip-reset movements for stable action recognition.

---

## Visual Interface  
Displays a live dashboard with real-time action states:
- OPENING THE BOTTLE  
- CLOSING THE BOTTLE  
- IDLE  

---

## Real-World Applications  

- Assembly line verification  
- Safety and compliance monitoring  
- Workforce training analytics  
- Robotics and teleoperation systems  

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


In [2]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [3]:
%pip install tqdm

from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['WhatsApp Video 2026-05-13 at 1.30.45 PM.mp4'],
    total_images=30,
    out_dir="dataset_Frames",
    jpg_quality=100,
    seed=42
)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[✓] Extracted 30 frames to folder: dataset_Frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [7]:
import json
import os
import shutil
import random
from pathlib import Path

# --- CONFIGURATION ---
base_path = r'D:\Desktop\Desk\Labellerr Github Projects\Use_Case_Projects\bottle open egocentric'
json_path = os.path.join(base_path, 'export-#50fbd0b3-d2d2-429b-bee4-888ca5048014.json')
images_src = os.path.join(base_path, 'dataset_Frames')
output_path = os.path.join(base_path, 'yolo_dataset')

# Ratios for your 30 frames
ratios = {'train': 0.8, 'val': 0.1, 'test': 0.1}

# Create Folder Structure
for split in ratios.keys():
    os.makedirs(os.path.join(output_path, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_path, 'labels', split), exist_ok=True)

# Generate data.yaml (formatted for YOLO11-Pose)
formatted_path = output_path.replace('\\', '/')
yaml_content = f"""
path: {formatted_path}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: bottle
  1: cap
  2: Hands

kpt_shape: [21, 3]
"""
with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
    f.write(yaml_content.strip())

# Load COCO JSON
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Build category map (Ensures 0: bottle, 1: cap, 2: Hands)
cat_map = {}
for cat in coco_data['categories']:
    name = cat['name'].lower()
    if 'bottle' in name: cat_map[cat['id']] = 0
    elif 'cap' in name: cat_map[cat['id']] = 1
    elif 'hand' in name: cat_map[cat['id']] = 2

images = {img['id']: img for img in coco_data['images']}
annotations = coco_data['annotations']

# Shuffle and Split IDs
img_ids = list(images.keys())
random.seed(42)
random.shuffle(img_ids)

train_idx = int(len(img_ids) * ratios['train'])
val_idx = train_idx + int(len(img_ids) * ratios['val'])

def convert_to_yolo(ann, img_w, img_h):
    cls_id = cat_map.get(ann['category_id'], -1)
    if cls_id == -1: return None

    line = [str(cls_id)]
    has_data = False
    
    # 1. Add Segmentation (Polygons for all classes)
    if 'segmentation' in ann and ann['segmentation']:
        poly = ann['segmentation'][0]
        if len(poly) > 4:
            for i in range(0, len(poly), 2):
                line.append(f"{poly[i]/img_w:.6f}")
                line.append(f"{poly[i+1]/img_h:.6f}")
            has_data = True
            
    # 2. Add Keypoints (ONLY for Hands class - ID 2)
    if cls_id == 2 and 'keypoints' in ann and ann['keypoints']:
        kpts = ann['keypoints']
        for i in range(0, len(kpts), 3):
            line.append(f"{kpts[i]/img_w:.6f}")
            line.append(f"{kpts[i+1]/img_h:.6f}")
            line.append(str(kpts[i+2]))
        has_data = True
            
    return " ".join(line) if has_data else None

# Process Files
for i, img_id in enumerate(img_ids):
    img_info = images[img_id]
    filename = img_info['file_name']
    split = 'train' if i < train_idx else 'val' if i < val_idx else 'test'
    
    src_img = os.path.join(images_src, filename)
    if os.path.exists(src_img):
        shutil.copy(src_img, os.path.join(output_path, 'images', split, filename))
        
        label_name = Path(filename).stem + ".txt"
        with open(os.path.join(output_path, 'labels', split, label_name), 'w') as lf:
            for ann in annotations:
                if ann['image_id'] == img_id:
                    yolo_line = convert_to_yolo(ann, img_info['width'], img_info['height'])
                    if yolo_line:
                        lf.write(yolo_line + "\n")

print(f"Conversion Done. Dataset and data.yaml are ready at: {base_path}")

Conversion Done. Dataset and data.yaml are ready at: D:\Desktop\Desk\Labellerr Github Projects\Use_Case_Projects\bottle open egocentric


# Load and Train YOLO Pose Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
from ultralytics import YOLO

# Using the x-pose model for your 21-point hand skeleton
model = YOLO('yolo11l-pose.pt') 

model.train(
    data='/kaggle/working/ydata.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='/kaggle/working/factory',
    name='train_run_11',
    device=0
)

# Pure YOLO11 Multi-Task Inference 
This script performs frame-by-frame inference using a single custom-trained **YOLO11 multi-task model**.  
It simultaneously predicts:
- Instance segmentation masks  
- Hand pose keypoints  

without relying on external tracking frameworks like MediaPipe.

---

## Core Workflow  

### Unified Model Loading  
Loads the custom `best.pt` weights to detect bottles, caps, and 21-point hand skeletons in one inference pass.

### Video Stream Processing  
Uses OpenCV to:
- Read input video  
- Extract FPS and frame dimensions  
- Create the output video writer  

### Native Prediction & Rendering  
Processes frames with `conf=0.5` and uses `.plot()` to automatically render:
- Segmentation masks  
- Bounding boxes  
- Hand skeletons  


---

In [ ]:
import cv2
from ultralytics import YOLO

# 1. Load your newly trained custom model
# This single model now natively predicts bottle, cap, and the 21 hand keypoints
model = YOLO('/kaggle/working/egobottle/train_run_seg_14/weights/best.pt')

# 2. Video Processing Setup
video_path = '/kaggle/input/datasets/aaryanaggarwal5040/ego-bottle/WhatsApp Video 2026-05-13 at 1.30.45 PM.mp4'
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter('/kaggle/working/pure_custom_inference.mp4', 
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print(f"Running raw inference using ONLY your custom model at {fps} FPS...")

while cap.isOpened():
    success, frame = cap.read()
    if not success: 
        break

    # Run inference directly through your trained weights
    results = model(frame, conf=0.5, verbose=False)
    
    # Ultralytics .plot() automatically handles drawing your custom bottle/cap masks 
    # AND the 21 hand keypoints exactly as it learned them during training!
    annotated_frame = results[0].plot()

    # Save frame to output video
    out.write(annotated_frame)

# Cleanup
cap.release()
out.release()
print("Done! View your true custom model predictions at: /kaggle/working/pure_custom_inference.mp4")

# Custom Model with Activity Refresh

## Overview  
This script detects bottle actions (**OPENING, CLOSING, IDLE**) using landmarks from a custom-trained **YOLO11 multi-task model**.  
It includes an activity-refresh mechanism that automatically resets the system to **IDLE** when twisting stops.

---

## Core Workflow  

### Angular Feature Extraction  
Tracks the vector between:
- Thumb Tip (keypoint 4)  
- Index Finger Tip (keypoint 8)  

The angle is calculated across a moving frame window to measure rotational motion.

---

### Intent-Lock Mechanism  
- **Opening** → Anticlockwise rotation (`< -12°`)  
- **Closing** → Clockwise rotation (`> +12°`)  

Once detected, the action state is locked to avoid false changes during grip readjustments.

---

### Active Refresh & Timeout  
If finger movement remains below a small threshold for over **0.6 seconds**, the system:
- Clears the action lock  
- Resets the state to **IDLE**

---

### Visual Rendering  
Displays a live dashboard with:
- Green → Opening  
- Red → Closing  
- White → Idle  

Rendered in a high-contrast UI block for clear visibility.

---

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque

# 1. Load your fully custom multi-task model
model = YOLO('/kaggle/working/egobottle/train_run_seg_14/weights/best.pt')

# 2. Logic Variables for Rotation & Timeout Tracking
angle_history = deque(maxlen=20)
current_action = "IDLE"
action_locked = False
no_movement_counter = 0  # Tracks consecutive frames with no active twisting
hand_lost_counter = 0

def calculate_angle(p1, p2):
    """Calculates angle between Thumb Tip and Index Tip."""
    return np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))

# 3. Video Setup
video_path = '/kaggle/input/datasets/aaryanaggarwal5040/ego-bottle/WhatsApp Video 2026-05-13 at 1.30.45 PM.mp4'
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter('/kaggle/working/custom_model_activity_detection.mp4', 
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("Processing video with native tracking & activity refresh logic...")

while cap.isOpened():
    success, frame = cap.read()
    if not success: 
        break

    # Run native inference
    results = model(frame, conf=0.5, verbose=False)
    annotated_frame = results[0].plot()
    
    # Check if custom model predicted hand keypoints
    if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
        hand_lost_counter = 0
        kpts = results[0].keypoints.data[0].cpu().numpy()
        
        if len(kpts) > 8:
            thumb_x, thumb_y = kpts[4][0], kpts[4][1]
            index_x, index_y = kpts[8][0], kpts[8][1]
            
            if thumb_x > 0 and index_x > 0:
                curr_angle = calculate_angle((thumb_x, thumb_y), (index_x, index_y))
                angle_history.append(curr_angle)

                if len(angle_history) >= 10:
                    # Look at recent short-term movement delta (e.g., last 5 frames) to find active stalls
                    recent_delta = abs(angle_history[-1] - angle_history[-5])
                    
                    # Net change across the whole window for intent classification
                    total_delta = angle_history[-1] - angle_history[0]

                    # --- ACTIVE REFRESH TRIGGER ---
                    # If the fingers stay within 2 degrees of movement, increment stall counter
                    if recent_delta < 2.0:
                        no_movement_counter += 1
                    else:
                        no_movement_counter = 0 # Active movement resets the counter

                    # If stalled for too long (e.g., ~0.6 seconds of zero twisting), unlock state
                    if no_movement_counter > int(fps * 0.6):
                        action_locked = False
                        current_action = "IDLE"

                    # --- INTENT SETTING ---
                    if not action_locked:
                        if total_delta < -12:    # Significant anticlockwise movement
                            current_action = "OPENING THE BOTTLE"
                            action_locked = True
                            no_movement_counter = 0
                        elif total_delta > 12:   # Significant clockwise movement
                            current_action = "CLOSING THE BOTTLE"
                            action_locked = True
                            no_movement_counter = 0
    else:
        # Hard reset fallback if hand completely disappears
        hand_lost_counter += 1
        if hand_lost_counter > fps:
            action_locked = False
            current_action = "IDLE"
            angle_history.clear()
            no_movement_counter = 0

    # --- UI OVERLAY ---
    cv2.rectangle(annotated_frame, (5, 5), (480, 65), (0, 0, 0), -1)
    
    if "OPENING THE BOTTLE" in current_action:
        text_color = (0, 255, 0) # Green
    elif "CLOSING THE BOTTLE" in current_action:
        text_color = (0, 0, 255) # Red
    else:
        text_color = (255, 255, 255) # White
        
    cv2.putText(annotated_frame, current_action, (15, 45), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, text_color, 3)

    out.write(annotated_frame)

cap.release()
out.release()
print("Process Complete! Video saved to /kaggle/working/custom_model_activity_detection.mp4")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
